In [2]:
import pandas as pd
from tqdm import tqdm
import seaborn as sns

/ebio/abt6/lvaness/anaconda3/envs/jupyter/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.1
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


## Get general statistics

In [3]:
ref=pd.read_pickle(f"../sqanti/tama_augustus_sqanti_full.pkl")


In [6]:
ref.loc[(ref["accession"]=="at6923")&(ref["gene"]=="G792")]

,gene,isoform,accession,associated_gene,associated_transcript,exons,length,isoform_read_support,gene_read_support,structural_category,subcategory,Sqanti_ORF_seq
61569,G792,G792.1,at6923,g1065,novel,8,2725,1,46,novel_not_in_catalog,at_least_one_novel_splicesite,MDLKENSFEHMGIHLGEIGIVLFPWFLWLSLFLSCGYAAITISSPL...
61570,G792,G792.10,at6923,g1065,g1065.t1,7,2388,1,46,full-splice_match,alternative_3end5end,MLPYSSLMYNLATGEKRVLSSWKSHTDPSPGDFVVRLTPQVPAQIV...
61571,G792,G792.11,at6923,g1065,g1065.t1,7,2515,1,46,full-splice_match,alternative_3end5end,MLPYSSLMYNLATGEKRVLSSWKSHTDPSPGDFVVRLTPQVPAQIV...
61572,G792,G792.12,at6923,g1065,g1065.t1,7,2630,2,46,full-splice_match,alternative_3end5end,MGIHLGEIGIVLFPWFLWLSLFLSCGYAAITISSPLTLGQTLSSPG...
61573,G792,G792.13,at6923,g1065,g1065.t1,7,2684,4,46,full-splice_match,alternative_3end5end,MGIHLGEIGIVLFPWFLWLSLFLSCGYAAITISSPLTLGQTLSSPG...
61574,G792,G792.14,at6923,g1065,g1065.t1,7,2770,9,46,full-splice_match,alternative_3end5end,MGIHLGEIGIVLFPWFLWLSLFLSCGYAAITISSPLTLGQTLSSPG...
61575,G792,G792.15,at6923,g1065,novel,8,2697,2,46,novel_not_in_catalog,at_least_one_novel_splicesite,MDLKENSFEHMGIHLGEIGIVLFPWFLWLSLFLSCGYAAITISSPL...
61576,G792,G792.16,at6923,g1065,novel,1,3477,1,46,genic,mono-exon,MGIHLGEIGIVLFPWFLWLSLFLSCGYAAITISSPLTLGQTLSSPG...
61577,G792,G792.17,at6923,g1065,g1065.t1,7,2840,1,46,full-splice_match,alternative_3end5end,MGIHLGEIGIVLFPWFLWLSLFLSCGYAAITISSPLTLGQTLSSPG...
61578,G792,G792.18,at6923,g1065,g1065.t1,7,2852,2,46,full-splice_match,alternative_3end5end,MGIHLGEIGIVLFPWFLWLSLFLSCGYAAITISSPLTLGQTLSSPG...


In [834]:
names=["tama_augustus","pb_augustus","tama_liftoff","pb_liftoff"]
runs=["../sqanti/tama_augustus_sqanti_full.pkl","../sqanti/pb_augustus_sqanti_full.pkl","../sqanti/tama_liftoff_sqanti_full.pkl","../sqanti/pb_liftoff_sqanti_full.pkl"]

rs_acc=pd.DataFrame()
rs_full_acc=pd.DataFrame()
rs_run=pd.DataFrame()
rs_run_acc=pd.DataFrame()

for name, run in zip(names,runs):
    # load full df
    ref=pd.read_pickle(f"{run}")
    # add new column
    ref["cat_assign"]="None"
    # and fill new colum based on 4 categories
    ref.loc[(ref["structural_category"] != "antisense") & (ref["structural_category"] != "fusion") & (ref["structural_category"] != "intergenic"), "cat_assign"]="one2one"
    ref.loc[(ref["structural_category"] == "antisense"), "cat_assign"]="antisense"
    ref.loc[(ref["structural_category"] == "fusion"), "cat_assign"]="fusion"
    ref.loc[(ref["structural_category"] == "intergenic") | (ref["structural_category"] == "genic_intron"), "cat_assign"]="novel"
    
    # in some cases, transcripts were reported as one2one even though the reads are spanning two genes.
    # In these cases, we have "fusion reads", but splicing results in perfect cds for one gene.
    # filter them and add replace the assignment
    ref["association"] = ref["associated_gene"].str.split("_").str.len()
    ref.loc[(ref["cat_assign"]=="one2one")&(ref["association"]>1), "cat_assign"] = "fusion"
    
    # add gene read support
    ref["cat_read_support"]=ref.groupby(by=["cat_assign","accession","gene"])["isoform_read_support"].transform("sum")

    #get read support per accession
    acc_read_contrib=ref[["gene","accession","cat_assign","cat_read_support"]]
    acc_read_contrib=acc_read_contrib.drop_duplicates()
    acc_read_contrib=acc_read_contrib.groupby(by=["accession"])["cat_read_support"].apply("sum").to_frame()
    acc_read_contrib.columns=[f"{name}_reads"]
    rs_acc=pd.concat([rs_acc, acc_read_contrib], axis=1)
    
    # get full rs table (reads-isoforms-genes)
    acc_full=ref[["gene","isoform","accession","cat_assign","cat_read_support"]]
    acc_isos=acc_full.groupby(by=["accession"]).apply(lambda x: len(x)).to_frame()
    acc_isos.columns=[f"{name}_isoforms"]
    acc_genes=acc_full.groupby(by=["accession"])["gene"].unique().apply(lambda x: len(x)).to_frame()
    acc_genes.columns=[f"{name}_genes"]
    rs_full_acc=pd.concat([rs_full_acc,acc_read_contrib,acc_isos,acc_genes], axis=1)

    # get read support per category
    run_read_contrib=ref[["gene","accession","cat_assign","cat_read_support"]]
    run_read_contrib=run_read_contrib.drop_duplicates()
    run_read_contrib=run_read_contrib.groupby(by=["cat_assign"])["cat_read_support"].apply("sum").to_frame()
    run_read_contrib.columns=[f"{name}"]
    rs_run=pd.concat([rs_run,run_read_contrib.T])
    
    #get read support per category per accession
    run_read_contrib_acc=ref[["gene","accession","cat_assign","cat_read_support"]]
    run_read_contrib_acc=run_read_contrib_acc.drop_duplicates()
    run_read_contrib_acc=run_read_contrib_acc.groupby(by=["cat_assign","accession"])["cat_read_support"].apply("sum").to_frame()
    run_read_contrib_acc.columns=["reads"]
    run_read_contrib_acc.reset_index(inplace=True)
    run_read_contrib_acc["run"]=f"{name}"
    rs_run_acc=pd.concat([rs_run_acc, run_read_contrib_acc])

In [835]:
rs_run

cat_assign,antisense,fusion,novel,one2one
tama_augustus,45745,64275,62109,9710849
pb_augustus,47620,72707,64304,10058948
tama_liftoff,41734,29847,120269,9691128
pb_liftoff,42853,38541,124658,10037527


In [836]:
rs_acc.to_csv("read_support_acc-vs-run.tsv", sep="\t")
rs_full_acc.to_csv("read_support-iso-gene_acc.tsv", sep="\t")
rs_run.to_csv("read_support_run-vs-category.tsv", sep="\t")
rs_run_acc.to_csv("read_support_run-vs-category_acc.tsv", sep="\t")